In [ ]:
from pathlib import Path
import os
import shutil

import pandas as pd
from IPython.display import display
from huggingface_hub import HfApi, hf_hub_download


In [ ]:
INPUT_DIR = "pyramid-forcing-30s/"
OUTPUT_DIR = "data/26Mar25-PyramidForcingView/videos/pyramid-forcing-30s"

In [ ]:
def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


def resolve_output_dir(output_dir: str) -> Path:
    raw = Path(output_dir)
    return raw if raw.is_absolute() else resolve_repo_root() / raw


HF_ENDPOINT = os.environ.get("HF_ENDPOINT", "https://hf-mirror.com")
REPO_ID = "kv-compression/Pyramid-Forcing-Experiments"
OUTPUT_PATH = resolve_output_dir(OUTPUT_DIR)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
{
    "input_dir": INPUT_DIR,
    "output_dir": str(OUTPUT_PATH),
}


In [ ]:
api = HfApi(endpoint=HF_ENDPOINT)
repo_files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
target_files = sorted(
    path
    for path in repo_files
    if path.startswith(INPUT_DIR) and Path(path).suffix.lower() in {".mp4", ".mov", ".avi", ".mkv", ".webm"}
)
if not target_files:
    raise FileNotFoundError(f"No video files matched dataset group: {INPUT_DIR}")

rows = []
for repo_path in target_files:
    destination_path = OUTPUT_PATH / Path(repo_path).name
    cached_path = Path(
        hf_hub_download(
            repo_id=REPO_ID,
            repo_type="dataset",
            filename=repo_path,
            endpoint=HF_ENDPOINT,
        )
    )
    shutil.copy2(cached_path, destination_path)
    rows.append({"repo_path": repo_path, "local_path": str(destination_path)})

result_df = pd.DataFrame(rows)
display(result_df)
print(f"Downloaded {len(result_df)} files into {OUTPUT_PATH}")
